In [1]:
!git clone https://github.com/avaturn-live/avtr-1.git
%cd avtr-1
!curl -fsSL https://pixi.sh/install.sh | sh
import os
os.environ["PATH"] = "/root/.pixi/bin:" + os.environ["PATH"]
!pixi install

Cloning into 'avtr-1'...
remote: Enumerating objects: 174, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 174 (delta 17), reused 28 (delta 15), pack-reused 112 (from 1)
Receiving objects: 100% (174/174), 708.34 KiB | 19.68 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/avtr-1
This script will automatically download and install Pixi (latest) for you.
Getting it from this url: https://github.com/prefix-dev/pixi/releases/latest/download/pixi-x86_64-unknown-linux-musl.tar.gz
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 31.1M  100 31.1M    0     0  63.6M      0 --:--:-- --:--:-- --:--:-- 63.6M
The 'pixi' binary is installed into '/roo

In [2]:
from huggingface_hub import login
login(token="", add_to_git_credential=False)
print("HF auth complete")

HF auth complete


In [3]:
!pixi run python -m ensurepip --upgrade
!pixi run python -m pip install "onnxruntime-gpu>=1.20"

 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
   ╭─[/kaggle/working/avtr-1/pixi.toml:7:1]
 6 │     
 7 │ ╭─▶ [system-requirements]
 8 │ ├─▶ cuda = "12.8"
   · ╰──── declare these on the `platforms` entries instead
 9 │     
   ╰────
  help: e.g. platforms = [{ name = "my-linux-64", platform = "linux-64", cuda
        = "12.8" }]

⠁ activating environment                                                                 Looking in links: /tmp/tmp_0mc0hfx
Processing /tmp/tmp_0mc0hfx/pip-23.2.1-py3-none-any.whl
 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
   ╭─[/kaggle/working/avtr-1/pixi.toml:7:1]
 6 │     
 7 │ ╭─▶ [system-requirements]
 8 │ ├─▶ cuda = "12.8"
   · ╰──── declare these on the `platforms` entries instead
 9 │     
   ╰────
  help: e.g. platforms = [{ name = 

In [4]:
!pixi run python scripts/download_artifacts.py

 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
   ╭─[/kaggle/working/avtr-1/pixi.toml:7:1]
 6 │     
 7 │ ╭─▶ [system-requirements]
 8 │ ├─▶ cuda = "12.8"
   · ╰──── declare these on the `platforms` entries instead
 9 │     
   ╰────
  help: e.g. platforms = [{ name = "my-linux-64", platform = "linux-64", cuda
        = "12.8" }]

⠁ activating environment                                                                 Storage: /kaggle/working/avtr-1/artifacts/main
ditto_onnx/landmark106.onnx:   0%|                  | 0.00/4.96M [00:00<?, ?B/s]
ditto_onnx/appearance_extractor.onnx:   0%|         | 0.00/3.36M [00:00<?, ?B/s]

ditto_onnx/motion_extractor.onnx:   0%|              | 0.00/113M [00:00<?, ?B/s]


ditto_onnx/landmark106.onnx: 100%|█████████| 4.96M/4.96M [00:00<00:00, 12.2MB/s]
ditto_onnx/insightface_det.onnx:   0%|              | 0.00/16.9M [00:00<?, ?B/s]
ditto_onnx/app

In [5]:
import glob, shutil
from pathlib import Path

REPO = Path("/kaggle/working/avtr-1")
# leave as None to auto-search /kaggle/input, or set e.g. "/kaggle/input/avtr1-production-engines"
ENGINES_DATASET = None

DEST = {
    "avtr1_encode_fp16.engine":   "artifacts/main/speech2motion_runtime_artifacts_cc",
    "avtr1_decode_fp16.engine":   "artifacts/main/speech2motion_runtime_artifacts_cc",
    "hubert_lbs_fp16.engine":     "artifacts/main/speech2motion_runtime_artifacts_cc",
    "decoder_b5_fp16.engine":     "artifacts/main/renderer_runtime_artifacts_cc",
    "warp_network_b5_fp16.engine":"artifacts/main/renderer_runtime_artifacts_cc",
    "modnet_b5_fp16.engine":      "artifacts/main/renderer_runtime_artifacts_cc",
    "stitch_network_b5_fp16.engine":"artifacts/main/renderer_runtime_artifacts_cc",
}

search_root = ENGINES_DATASET or "/kaggle/input"
missing = []
for name, rel in DEST.items():
    hits = glob.glob(f"{search_root}/**/{name}", recursive=True)
    if not hits:
        missing.append(name); print("MISSING ", name); continue
    dst_dir = REPO / rel
    dst_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(hits[0], dst_dir / name)
    print(f"OK  {name}  <-  {hits[0]}")

if missing:
    raise FileNotFoundError(f"Not found under {search_root}: {missing}")
print("\nAll 7 engines installed.")

OK  avtr1_encode_fp16.engine  <-  /kaggle/input/datasets/henil2132/avtar-engines/speech2motion/avtr1_encode_fp16.engine
OK  avtr1_decode_fp16.engine  <-  /kaggle/input/datasets/henil2132/avtar-engines/speech2motion/avtr1_decode_fp16.engine
OK  hubert_lbs_fp16.engine  <-  /kaggle/input/datasets/henil2132/avtar-engines/speech2motion/hubert_lbs_fp16.engine
OK  decoder_b5_fp16.engine  <-  /kaggle/input/datasets/henil2132/avtar-engines/renderer/decoder_b5_fp16.engine
OK  warp_network_b5_fp16.engine  <-  /kaggle/input/datasets/henil2132/avtar-engines/renderer/warp_network_b5_fp16.engine
OK  modnet_b5_fp16.engine  <-  /kaggle/input/datasets/henil2132/avtar-engines/renderer/modnet_b5_fp16.engine
OK  stitch_network_b5_fp16.engine  <-  /kaggle/input/datasets/henil2132/avtar-engines/renderer/stitch_network_b5_fp16.engine

All 7 engines installed.


In [6]:
%cd /kaggle/working/avtr-1
!ls -lh artifacts/main/speech2motion_runtime_artifacts_cc/ artifacts/main/renderer_runtime_artifacts_cc/
# plugin the renderer needs at runtime; materialize if download left it empty/missing
!pixi run python -c "from avtr1_renderer.avtr1_artifact_manager import get_artifact_manager as g; print('plugin:', g().get_artifact_path('warp_plugin'))"
!ls -lh artifacts/main/renderer_runtime_artifacts/libgrid_sample_3d_plugin.so

/kaggle/working/avtr-1
artifacts/main/renderer_runtime_artifacts_cc/:
total 223M
-rw-rw-rw- 1 root root 108M Nov  6 16:03 decoder_b5_fp16.engine
-rw-rw-rw- 1 root root  16M Nov  6 16:03 modnet_b5_fp16.engine
-rw-rw-rw- 1 root root 471K Nov  6 16:03 stitch_network_b5_fp16.engine
-rw-rw-rw- 1 root root  99M Nov  6 16:03 warp_network_b5_fp16.engine

artifacts/main/speech2motion_runtime_artifacts_cc/:
total 907M
-rw-rw-rw- 1 root root 279M Nov  6 16:03 avtr1_decode_fp16.engine
-rw-rw-rw- 1 root root  21M Nov  6 16:03 avtr1_encode_fp16.engine
-rw-rw-rw- 1 root root 608M Nov  6 16:03 hubert_lbs_fp16.engine
 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
   ╭─[/kaggle/working/avtr-1/pixi.toml:7:1]
 6 │     
 7 │ ╭─▶ [system-requirements]
 8 │ ├─▶ cuda = "12.8"
   · ╰──── declare these on the `platforms` entries instead
 9 │     
   ╰────
  help: e.g. platforms = [{ name = "my-linux-64",

In [11]:
%%writefile /kaggle/working/make_normalizer.py
import torch
from safetensors.torch import save_file
from avtr1_renderer.avtr1_artifact_manager import get_artifact_manager, get_storage_root

NAMES = ("motion_offset", "motion_scale", "offset_so3", "scale_so3",
         "offset_kp", "scale_kp", "offset_exp", "scale_exp", "lipsync_coords")

mgr  = get_artifact_manager()
ckpt = mgr.get_artifact_path("avtr1_scripted")          # build_artifacts/avtr1.scripted.pt (from Cell 4)
out  = get_storage_root() / "avtr1_normalizer.safetensors"

dev = "cuda" if torch.cuda.is_available() else "cpu"
scripted = torch.jit.load(str(ckpt), map_location=dev).eval()

tensors = {}
for n in NAMES:
    if not hasattr(scripted, n):
        raise RuntimeError(f"checkpoint missing buffer '{n}' (needs a newer scripted .pt)")
    tensors[n] = getattr(scripted, n).detach().cpu().contiguous()

out.parent.mkdir(parents=True, exist_ok=True)
save_file(tensors, str(out))
print("wrote", out, {k: tuple(v.shape) for k, v in tensors.items()})

Writing /kaggle/working/make_normalizer.py


In [12]:
%cd /kaggle/working/avtr-1
!pixi run python /kaggle/working/make_normalizer.py
!ls -lh artifacts/main/avtr1_normalizer.safetensors

/kaggle/working/avtr-1
 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
   ╭─[/kaggle/working/avtr-1/pixi.toml:7:1]
 6 │     
 7 │ ╭─▶ [system-requirements]
 8 │ ├─▶ cuda = "12.8"
   · ╰──── declare these on the `platforms` entries instead
 9 │     
   ╰────
  help: e.g. platforms = [{ name = "my-linux-64", platform = "linux-64", cuda
        = "12.8" }]

⠁ activating environment                                                                 wrote /kaggle/working/avtr-1/artifacts/main/avtr1_normalizer.safetensors {'motion_offset': (42,), 'motion_scale': (42,), 'offset_so3': (3,), 'scale_so3': (3,), 'offset_kp': (21, 3), 'scale_kp': (21, 3), 'offset_exp': (21, 3), 'scale_exp': (21, 3), 'lipsync_coords': (39,)}
-rw-r--r-- 1 root root 2.3K Nov  6 16:15 artifacts/main/avtr1_normalizer.safetensors


In [13]:
# audio
!wget -O /kaggle/working/speech.mp3 "https://raw.githubusercontent.com/bala-ceg/doctor-patient-indic-speech-dataset/main/audio/hindi/convo_001.mp3"

# optional: add a custom avatar face as a 1920x1080 reference frame named "myavatar"
from PIL import Image
src = "/kaggle/input/datasets/henil2132/docface/0599dc3d-1479-4604-8584-21a7ca6fd54c.jpg"
img = Image.open(src).convert("RGBA")
canvas = Image.new("RGBA", (1920, 1080), (0, 0, 0, 0))
img.thumbnail((1080, 1080))
canvas.paste(img, ((1920 - img.width)//2, (1080 - img.height)//2), img)
canvas.save("artifacts/main/avatars_artifacts/reference_frames/myavatar.png")
print("avatar saved")

--2026-11-06 16:15:36--  https://raw.githubusercontent.com/bala-ceg/doctor-patient-indic-speech-dataset/main/audio/hindi/convo_001.mp3
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1041908 (1017K) [audio/mpeg]
Saving to: ‘/kaggle/working/speech.mp3’

/kaggle/working/spe 100%[===================>]   1017K  --.-KB/s    in 0.009s  

2026-11-06 16:15:36 (106 MB/s) - ‘/kaggle/working/speech.mp3’ saved [1041908/1041908]

avatar saved


In [16]:
%cd /kaggle/working/avtr-1
!pixi run generate_offline \
    --speech /kaggle/input/datasets/henil2132/audio-doc/download-clien.mp3 \
    --avatar myavatar \
    --bg plain_white \
    --out /kaggle/working/avatar_out.mp4

/kaggle/working/avtr-1
 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
   ╭─[/kaggle/working/avtr-1/pixi.toml:7:1]
 6 │     
 7 │ ╭─▶ [system-requirements]
 8 │ ├─▶ cuda = "12.8"
   · ╰──── declare these on the `platforms` entries instead
 9 │     
   ╰────
  help: e.g. platforms = [{ name = "my-linux-64", platform = "linux-64", cuda
        = "12.8" }]

✨ Pixi task (generate_offline in default): python scripts/generate_offline.py --speech /kaggle/input/datasets/henil2132/audio-doc/download-clien.mp3 --avatar myavatar --bg plain_white --out /kaggle/working/avatar_out.mp4: (Run end-to-end offline generation (requires built engines))
⠁ activating environment                                                                 Audio: 117.52s (speech=set, listen=silence)
Loading pipeline for avatar 'myavatar'...
  (models are downloaded from HuggingFace on first run — this may take a few 

In [17]:
from IPython.display import Video, FileLink, display
display(Video("/kaggle/working/avatar_out.mp4", embed=True, width=800))
display(FileLink("/kaggle/working/avatar_out.mp4"))

/kaggle/working/avatar_out.mp4